# ASL Sign Language Interpreter
## 2. Training
ResNet18-basierter Klassifikator für das American Sign Language Alphabet (29 Klassen).

### Imports & Konfiguration

In [1]:
import os
import ssl

import torch
import matplotlib.pyplot as plt
from torch import nn, optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

DATA_DIR   = "../data/offline_composited"
MODEL_PATH = "../model_resnet18_asl_best.pth"
DEVICE     = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

NUM_CLASSES     = 29
IMG_SIZE        = 224
BATCH_SIZE      = 64
NUM_WORKERS     = 4
TRAIN_VAL_SPLIT = 0.8
NUM_EPOCHS      = 20
LR_HEAD         = 0.001
LR_FINETUNE     = 0.0001
UNFREEZE_EPOCH  = 5

print(f"Device: {DEVICE}")

Device: mps


### Daten laden

In [2]:
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Indizes für Train/Val-Split bestimmen
full_dataset = datasets.ImageFolder(root=DATA_DIR)
n = len(full_dataset)
n_train = int(n * TRAIN_VAL_SPLIT)
indices = torch.randperm(n).tolist()
train_indices, val_indices = indices[:n_train], indices[n_train:]

# Separate Datensätze mit unterschiedlichen Transforms
train_data = Subset(datasets.ImageFolder(root=DATA_DIR, transform=transform_train), train_indices)
val_data   = Subset(datasets.ImageFolder(root=DATA_DIR, transform=transform_val),   val_indices)

trainloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, persistent_workers=True)
valloader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=True)

print(f"Gesamtbilder:      {n}")
print(f"Trainingsbilder:   {len(train_data)}")
print(f"Validationsbilder: {len(val_data)}")

Gesamtbilder:      174000
Trainingsbilder:   139200
Validationsbilder: 34800


### Modell

In [3]:
ssl._create_default_https_context = ssl._create_unverified_context

model = models.resnet18(weights="DEFAULT")

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(in_features=512, out_features=NUM_CLASSES)
model = model.to(DEVICE)

print(f"Letzte Schicht: {model.fc}")
print(f"Modell auf:     {DEVICE}")

Letzte Schicht: Linear(in_features=512, out_features=29, bias=True)
Modell auf:     mps


### Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCH)
best_val_acc = 0.0

history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    if epoch == UNFREEZE_EPOCH:
        print("Entfriere alle Schichten für Fine-Tuning mit geringerer Lernrate...")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LR_FINETUNE)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - UNFREEZE_EPOCH)

    model.train()
    running_loss = 0.0
    for images, labels in tqdm(trainloader, desc=f"Epoch {epoch+1:2d}/{NUM_EPOCHS} Training", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    model.eval()
    correct = total = 0
    val_running_loss = 0.0
    with torch.no_grad():
        for images, labels in tqdm(valloader, desc=f"Epoch {epoch+1:2d}/{NUM_EPOCHS} Validation", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            val_running_loss += criterion(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(trainloader)
    val_loss = val_running_loss / len(valloader)
    val_acc = correct / total
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | LR: {current_lr:.6f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"           -> Gespeichert (beste Val Acc: {best_val_acc*100:.2f}%)")

print("Training abgeschlossen!")

Epoch  1/20 Training:   0%|          | 0/2175 [00:04<?, ?it/s]

Epoch  1/20 Validation:   0%|          | 0/544 [00:04<?, ?it/s]

Epoch  1/20 | Train Loss: 2.1165 | Val Loss: 1.9019 | Val Acc: 47.64% | LR: 0.000905
           -> Gespeichert (beste Val Acc: 47.64%)


Epoch  2/20 Training:   0%|          | 0/2175 [00:00<?, ?it/s]

### Trainingsverlauf

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], marker="o", label="Train Loss")
ax1.plot(epochs, history["val_loss"], marker="o", label="Val Loss")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, [v * 100 for v in history["val_acc"]], marker="o", color="orange")
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Beste Val Acc: {max(history['val_acc'])*100:.2f}% (Epoch {history['val_acc'].index(max(history['val_acc']))+1})")